In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

## JSD - Model connection:

- I am looking for the connection between model performance and the metric values
- specifically this is to show if datasets with better jsd values result in better model performance
- Steps:
    - Get datasets with different metric values
    - Show that the values for the different metrics show roughly the same trends
    - Train a few models on the dataset
    - Look at the model performance on the whole allowed state-action space

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

from dmpe.data_management import DataPaths
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.models.models import NeuralEulerODECartpole

### checkout data:

In [ ]:
cart_pole_data_path = DataPaths().cs_experiments / "dmpe" / "cart_pole"

In [ ]:
experiment_results = []

for exp_id in get_experiment_ids(cart_pole_data_path):
    params, observations, actions, model = load_experiment_results(
        exp_id,
        cart_pole_data_path,
        NeuralEulerODECartpole
    )

    experiment_results.append(
        dict(
            exp_id=exp_id,
            params=params,
            observations=observations,
            actions=actions,
            model=model,
        )
    )

In [ ]:
data = experiment_results[0]

In [ ]:
data["observations"]

In [ ]:
plt.plot(data["actions"])

In [ ]:
data_points = jnp.concatenate([data["observations"][:-1], data["actions"]], axis=-1)

### evaluate model and data

- checkout model performance of the given models
- checkout jsd of the given datasets

In [ ]:
from dmpe.evaluation.data_evaluation import DataEvaluator,JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison
from dmpe.excitation.excitation_utils import soft_penalty
import exciting_environments as excenvs

In [ ]:
# def vector_field(t, y, args):
#     deflection, velocity, theta, omega = y
#     action, params = args
#     d_omega = (
#         params.g * jnp.sin(theta)
#         + jnp.cos(theta)
#         * (
#             (
#                 -action[0]
#                 - params.m_p * params.l * (omega**2) * jnp.sin(theta)
#                 + params.mu_c * jnp.sign(velocity)
#             )
#             / (params.m_c + params.m_p)
#         )
#         - (params.mu_p * omega) / (params.m_p * params.l)
#     ) / (params.l * (4 / 3 - (params.m_p * (jnp.cos(theta)) ** 2) / (params.m_c + params.m_p)))

#     d_velocity = (
#         action[0]
#         + params.m_p * params.l * ((omega**2) * jnp.sin(theta) - d_omega * jnp.cos(theta))
#         - params.mu_c * jnp.sign(velocity)
#     ) / (params.m_c + params.m_p)
#     d_theta = omega
#     d_deflection = velocity
#     d_y = d_deflection, d_velocity, d_theta, d_omega
#     return d_y

In [ ]:
def constr_function(data_point, max_value=1, penalty_order=2):
    penalties = jax.nn.relu(jnp.abs(data_point) - max_value)
    penalties = penalties**penalty_order

    penalty = jnp.sum(penalties)
    return jnp.squeeze(penalty)


class NodeModelWrapper(ModelWrapper):
    """Wraps a DMPE NODE model for comparison."""

    def step(self, obs, action, tau):
        return self.model.step(obs, action, tau)

    def gradient(self, obs, action):
        return self.model.func(obs, action)

    def rollout(self, init_obs, actions, tau):
        return self.model(init_obs, actions, tau)


class EnvWrapper(ModelWrapper):
    """Wraps an exciting_environments env for comparison.
    
    can you do this by differentiating the step function by time?
    """

    @eqx.filter_jit
    def step(self, obs, action, tau):
        state = self.model.generate_state_from_observation(obs, self.model.env_properties)
        next_obs, _ = self.model.step(state, action, self.model.env_properties)
        return next_obs

    @eqx.filter_jit
    def gradient(self, obs, action):
        # state = self.model.generate_state_from_observation(obs, self.model.env_properties)
        # physical_state = state.physical_state
        # args = (action, self.model.env_properties.static_params)

        # x0 = tuple(
        #     [
        #         physical_state.deflection,
        #         physical_state.velocity,
        #         physical_state.theta,
        #         physical_state.omega,
        #     ]
        # )
        # dxdt = vector_field(t=0, y=x0, args=args)
        # d_deflection = y[0]
        # d_velocity = y[1]
        # d_theta = y[2]
        # d_omega = y[3]

        # # normalize
        # d_obs = jnp.array([
        #     self.model.env_properties.physical_normalizations.deflection.normalize(d_deflection),
        #     self.model.env_properties.physical_normalizations.velocity.normalize(d_velocity),
        #     self.model.env_properties.physical_normalizations.theta.normalize(d_theta),
        #     self.model.env_properties.physical_normalizations.omega.normalize(d_omega),
        # ])
        
        # return d_obs
        raise NotImplementedError
        
    @eqx.filter_jit
    def rollout(self, init_obs, actions, tau):
        init_state = self.model.generate_state_from_observation(init_obs, self.model.env_properties)
        observations, _, _ = self.model.sim_ahead(init_state, actions, self.model.env_properties, tau, tau)
        return observations

In [ ]:
env_params = dict(
    batch_size=1,
    tau=2e-2,
    max_force=10,
    static_params={
        "mu_p": 0.002,
        "mu_c": 0.5,
        "l": 0.5,
        "m_p": 0.1,
        "m_c": 1,
        "g": 9.81,
    },
    physical_normalizations={
        "deflection": excenvs.utils.MinMaxNormalization(min=-2.4, max=2.4),
        "velocity": excenvs.utils.MinMaxNormalization(min=-8, max=8),
        "theta": excenvs.utils.MinMaxNormalization(min=-jnp.pi, max=jnp.pi),
        "omega": excenvs.utils.MinMaxNormalization(min=-8, max=8),
    },
    env_solver=diffrax.Tsit5(),
)
env = excenvs.make(
    env_id="CartPole-v0",
    batch_size=env_params["batch_size"],
    action_normalizations={
        "force": excenvs.utils.MinMaxNormalization(min=-env_params["max_force"], max=env_params["max_force"])
    },
    physical_normalizations=env_params["physical_normalizations"],
    static_params=env_params["static_params"],
    solver=env_params["env_solver"],
    tau=env_params["tau"],
)

wrapped_env = EnvWrapper(env)

In [ ]:
cart_pole_data_path = DataPaths().cs_experiments / "dmpe" / "cart_pole" / "old_results"

In [ ]:
experiment_results = []

for exp_id in get_experiment_ids(cart_pole_data_path):
    params, observations, actions, model = load_experiment_results(
        exp_id,
        cart_pole_data_path,
        NeuralEulerODECartpole
    )

    experiment_results.append(
        dict(
            exp_id=exp_id,
            params=params,
            observations=observations,
            actions=actions,
            model=model,
        )
    )

In [ ]:
jsd_values = []
model_error_values = []

for data in tqdm(experiment_results):
    data_points = jnp.concatenate([data["observations"][:-1], data["actions"]], axis=-1)
    wrapped_model = NodeModelWrapper(data["model"])

    data_evaluator = DataEvaluator(
        constraint_function=constr_function,
        data_dim=5,
        points_per_dim=10,
    )
    performance_metric_results = data_evaluator.get_metrics(data_points)

    model_evaluator = ModelEvaluator(
        constraint_function=constr_function,
        gt_model=wrapped_env,
        obs_dim=4,
        act_dim=1,
        validation_points_per_dim=10,
        tau=env.tau,
    )
    diff_map, metric = model_evaluator.default_metrics["pred_comp"](wrapped_model, model_evaluator.gt_model)

    jsd_values.append(performance_metric_results["jsd"].item())
    model_error_values.append(metric.item())    

In [ ]:
plt.plot(np.array(jsd_values)[np.argsort(jsd_values)])
plt.title("JSD values for datasets sorted min to max")

In [ ]:
plt.plot(np.array(model_error_values)[np.argsort(jsd_values)])
plt.title("model performance values for corresponding models sorted min-jsd to max-jsd")

### Detailed look at single example

In [ ]:
data = experiment_results[0]
data_points = jnp.concatenate([data["observations"][:-1], data["actions"]], axis=-1)
wrapped_model = NodeModelWrapper(data["model"])

In [ ]:
data_evaluator = DataEvaluator(
    constraint_function=constr_function,
    data_dim=5,
    points_per_dim=10,
)
data_evaluator.get_metrics(data_points)

In [ ]:
model_evaluator = ModelEvaluator(
    constraint_function=constr_function,
    gt_model=wrapped_env,
    obs_dim=4,
    act_dim=1,
    validation_points_per_dim=10,
    tau=env.tau,
)
diff_map, metric = model_evaluator.default_metrics["pred_comp"](wrapped_model, model_evaluator.gt_model)
metric

In [ ]:
diff_map

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

x = model_evaluator.constraint_data_space_grid[:, 0]
y = model_evaluator.constraint_data_space_grid[:, 1]
z = diff_map[:,0]

ax.plot_trisurf(x,y,z)

In [ ]:
pred_observations_env = wrapped_env.rollout(init_obs=observations[0, :], actions=actions[:100], tau=env.tau)
pred_observations_model = wrapped_model.rollout(init_obs=observations[0, :], actions=actions[:100], tau=env.tau)

In [ ]:
for env_pred, model_pred in zip(pred_observations_env.T, pred_observations_model.T): 
    plt.plot(env_pred)
    plt.plot(model_pred)
    plt.show()